# BCC Lattice Cleaving — Monte Carlo States Sweep (100^3, largest grain + full structure)

Visualizes 5 independent 3D Monte Carlo (Potts-model) grain-growth cases,
swept over the number of states `S`, each captured at a specific temporal
slice, all on a (100,100,100) domain -- the full cleaving pipeline (exact-
boundary clip + cleave, THEN Laplacian relaxation, THEN both shape-quality
optimization passes) applied to every case.

| Case | S (states) | Target tslice | Note |
|---|---|---|---|
| `S2_T2` | 2 | 2 | tslice=3 is unreachable -- see below |
| `S3_T4` | 3 | 4 | |
| `S4_T6` | 4 | 6 | |
| `S5_T10` | 5 | 10 | |
| `S10_T20` | 10 | 20 | |

**S=2 note**: tslice=3 was originally requested for S=2, but confirmed
directly (on both a small test domain and this actual (100,100,100) domain)
that with only 2 Potts states, this dashboard's simulation reaches "single
crystal achieved" -- the entire domain becomes ONE grain -- and the
simulation HALTS ENTIRELY by MC step 2, regardless of how many steps are
requested. tslice=3 therefore never exists for S=2 here; tslice=2 (the
actual last/only reachable slice) is used instead.

**Why this notebook loads results instead of computing them live**: this is
by far the largest scale this cleaving module has been run at. Diagnostic
runs before committing to the full sweep measured the S=3 case alone
cleaving to 23.37M tets (534s) and the S=10 case to 46.86M tets (904s) --
several times larger than any mesh previously validated in this module (the
largest prior case, in `outputs/cleaving/voronoi_tessellation/`, was 13.1M
tets). Running the full pipeline (cleave + relax + 2 optimize passes) live,
for all 5 cases, inside an interactive notebook would take hours per case.
Instead, `outputs/cleaving/mc_states_sweep/scripts/generate_mc_states_sweep.py`
runs the whole sweep ONCE as a batch job (saving each case's results as soon
as it finishes, not just at the end) and this notebook loads and visualizes
what it produced -- the same generate-once / view-many pattern already used
by `mc_S3_temporal/`, `mc_S2_independent_runs/`, and `voronoi_tessellation/`
in this same `outputs/cleaving/` directory.

For each case, two meshes are available: the ENTIRE final grain structure,
and a separate, compacted mesh of just its LARGEST (by volume) grain.

## 0. Imports and path setup

In [ ]:
import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Make upxo importable regardless of where the notebook is opened from
_NB_DIR    = os.path.abspath('')
_REPO_ROOT = os.path.abspath(os.path.join(_NB_DIR, '..', '..', '..', '..'))
_SRC_DIR   = os.path.join(_REPO_ROOT, 'src')

_OUT_ROOT = os.path.join(_SRC_DIR, 'upxo', 'outputs', 'cleaving', 'mc_states_sweep')
_DATA_DIR = os.path.join(_OUT_ROOT, 'data')
_DATA_LARGEST_DIR = os.path.join(_OUT_ROOT, 'data_largest_grain')
_IMG_DIR = os.path.join(_OUT_ROOT, 'images')
_IMG_LARGEST_DIR = os.path.join(_OUT_ROOT, 'images_largest_grain')
_SUMMARY_PATH = os.path.join(_OUT_ROOT, 'summary.txt')

print(f'output root: {_OUT_ROOT}')

## 1. Configuration — the 5 cases

In [ ]:
CASES = [
    dict(case_id='S2_T2', S=2, target_tslice=2),
    dict(case_id='S3_T4', S=3, target_tslice=4),
    dict(case_id='S4_T6', S=4, target_tslice=6),
    dict(case_id='S5_T10', S=5, target_tslice=10),
    dict(case_id='S10_T20', S=10, target_tslice=20),
]

_VTK_TETRA = 10


def to_grid(node_coords, tets):
    n_tets = len(tets)
    cells_ = np.hstack([np.full((n_tets, 1), 4, dtype=np.int64), tets]).ravel()
    cell_types = np.full(n_tets, _VTK_TETRA, dtype=np.uint8)
    return pv.UnstructuredGrid(cells_, cell_types, node_coords)


def load_structure(case_id):
    data = np.load(os.path.join(_DATA_DIR, f'{case_id}_mesh_final.npz'))
    grid = to_grid(data['node_coords'], data['tets'])
    unique_vals, rank = np.unique(data['tet_labels'], return_inverse=True)
    grid.cell_data['label_rank'] = rank
    return grid, len(unique_vals)


def load_largest_grain(case_id):
    data = np.load(os.path.join(_DATA_LARGEST_DIR, f'{case_id}_largest_grain.npz'))
    grid = to_grid(data['node_coords'], data['tets'])
    return grid, int(data['label']), float(data['volume'])

## 2. Generation summary

Timing, grain counts, tet counts, and volume-conservation checks for every case that has finished so far -- written incrementally by the generation script, so this reflects real progress even for a sweep still in flight.

In [ ]:
with open(_SUMMARY_PATH) as f:
    print(f.read())

## 3. Static preview gallery

The generation script renders a screenshot of both meshes for every case as it finishes. Fast, non-interactive overview of all 5 cases at once -- see Sections 4-5 below for the interactive, drag-the-plane versions.

In [ ]:
available = [c for c in CASES if os.path.exists(
    os.path.join(_IMG_DIR, f'{c["case_id"]}_structure.png'))]
print(f'{len(available)} / {len(CASES)} cases rendered so far: '
      f'{[c["case_id"] for c in available]}')

fig, axes = plt.subplots(len(available), 2, figsize=(11, 5 * len(available)))
if len(available) == 1:
    axes = axes[None, :]
for row, case in enumerate(available):
    case_id = case['case_id']
    structure_img = mpimg.imread(os.path.join(_IMG_DIR, f'{case_id}_structure.png'))
    largest_img = mpimg.imread(os.path.join(_IMG_LARGEST_DIR, f'{case_id}_largest_grain.png'))
    axes[row, 0].imshow(structure_img)
    axes[row, 0].set_title(f'{case_id} \u2014 full structure')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(largest_img)
    axes[row, 1].set_title(f'{case_id} \u2014 largest grain')
    axes[row, 1].axis('off')
fig.tight_layout()
plt.show()

## 4. Interactive: full grain structure — crinkle clip

One interactive window per finished case (colour by grain label, crinkle-clipped along x so the interior grain-boundary surface is visible -- every cut tet is kept whole rather than sliced through).

In [ ]:
def plot_structure_crinkle(case_id, S, target_tslice):
    grid, n_grains = load_structure(case_id)
    pvp = pv.Plotter()
    pvp.set_background('white')
    pvp.add_mesh_clip_plane(grid, normal='x', crinkle=True,
                             scalars='label_rank', cmap='tab20', show_edges=True,
                             scalar_bar_args={'title': 'grain label (rank)'})
    pvp.add_title(f'{case_id}  (S={S}, tslice={target_tslice})  '
                   f'grains={n_grains}  tets={grid.n_cells}', font_size=10)
    pvp.add_axes()
    pvp.show()


for case in CASES:
    path = os.path.join(_DATA_DIR, f'{case["case_id"]}_mesh_final.npz')
    if os.path.exists(path):
        plot_structure_crinkle(case['case_id'], case['S'], case['target_tslice'])
    else:
        print(f'{case["case_id"]}: not finished yet, skipping')

## 5. Interactive: largest grain only — crinkle clip

Same crinkle-clip view, restricted to just the largest (by volume) grain from each case -- already compacted to its own node/tet arrays by the generation script.

In [ ]:
def plot_largest_grain_crinkle(case_id, S, target_tslice):
    grid, label, volume = load_largest_grain(case_id)
    pvp = pv.Plotter()
    pvp.set_background('white')
    pvp.add_mesh_clip_plane(grid, normal='x', crinkle=True,
                             color='lightsteelblue', show_edges=True)
    pvp.add_title(f'{case_id}  (S={S}, tslice={target_tslice})  largest grain  '
                   f'label={label}  volume={volume:.1f}  tets={grid.n_cells}', font_size=10)
    pvp.add_axes()
    pvp.show()


for case in CASES:
    path = os.path.join(_DATA_LARGEST_DIR, f'{case["case_id"]}_largest_grain.npz')
    if os.path.exists(path):
        plot_largest_grain_crinkle(case['case_id'], case['S'], case['target_tslice'])
    else:
        print(f'{case["case_id"]}: not finished yet, skipping')